# Активность владельцев ПРАЙМ и отток подписок

Осацкий Александр - Проект 1

Проверю, связана ли активность владельца с отменой его подписки в следующем месяце. Бизнес-вопрос: можно ли использовать эту активность, чтобы выбирать аудиторию для проверки кампаний удержания?

Гипотезы:

1. Владельцы, которые не пользовались сервисами последние 30 дней, чаще отменяют подписку в следующие 30 дней.
2. Среди активных владельцев пользователи одного сервиса уходят чаще, чем пользователи нескольких.

Основной период - июль. На данных июня проверю, повторяется ли результат.


## 1. Выборка и периоды

Беру свой сегмент из ДЗ-2: клиентов, которые покупали через `pos` в марте 2026 года. Выводы относятся к этому сегменту. Канал покупки `pos` и канал привлечения клиента — разные признаки.

Одна строка — одна подписка на дату наблюдения. Оставляю действующие подписки со стажем от 30 дней: у владельцев должен быть полный месяц для использования сервисов.

Сначала смотрю использование за предыдущие 30 дней, затем — отмены за следующие 30 дней. Так отмена не станет причиной низкой активности, которой мы пытаемся объяснить уход.

| Дата наблюдения | Период использования | Период отмен |
|---|---|---|
| 1 июня | 2–31 мая | 1–30 июня |
| 1 июля | 1–30 июня | 1–30 июля |

Все даты — в UTC. Начало каждого окна включается, правая граница — нет.

Изначально рассматривал и 1 мая, но в источнике не было сессий за 14–15 апреля по всей базе. Исключил этот период, чтобы пропуски данных не выглядели как отсутствие использования.


## 2. Подключение и даты

Использую подключение из `prime.config`. Данные для подключения хранятся в `.env`.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sqlalchemy import text
from prime.config import get_engine


In [ ]:
engine = get_engine()


In [ ]:
def q(sql: str, **params) -> pd.DataFrame:
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()

print("Подключились как:", q("select current_user as login").iloc[0, 0])


Условие своего сегмента получаю из таблицы `assignments`.


In [ ]:
assignment = q("select client_filter from prime.assignments where login = current_user")
CLIENT_FILTER = assignment.iloc[0, 0]
print("Сегмент:")
print(CLIENT_FILTER)


In [ ]:
SEGMENT_SQL = f"select client_id from prime.clients where {CLIENT_FILTER}"

CUTOFFS = pd.DatetimeIndex(["2026-06-01", "2026-07-01"], tz="UTC")
PRIMARY_CUTOFF = pd.Timestamp("2026-07-01", tz="UTC")
LOOKBACK = pd.Timedelta(days=30)
FOLLOWUP = pd.Timedelta(days=30)

windows = pd.DataFrame({"cutoff": CUTOFFS})
windows["activity_start"] = windows["cutoff"] - LOOKBACK
windows["outcome_end"] = windows["cutoff"] + FOLLOWUP
windows


## 3. Проверка полноты данных

Проверю, какие даты есть в источниках. Отдельно посмотрю сессии по каждому дню нужного периода: отсутствие использования у одного человека нормально, а отсутствие записей по всей базе может означать пропуск данных.


In [ ]:
coverage = q("""
    select 'service_usage' as source,
           min(started_at) as first_event, max(started_at) as last_event
    from prime.service_usage
    union all
    select 'payments', min(paid_at), max(paid_at) from prime.payments
    union all
    select 'transactions', min(occurred_at), max(occurred_at) from prime.transactions
    union all
    select 'subscription_cancellations', min(ended_at), max(ended_at)
    from prime.subscriptions
""")
coverage


In [ ]:
daily_source = q("""
    select started_at::date as day,
           count(*) as rows,
           count(*) filter (where ended_at > started_at) as valid_rows
    from prime.service_usage
    where started_at >= :a and started_at < :b
    group by 1 order by 1
""", a=(CUTOFFS.min() - LOOKBACK).to_pydatetime(), b=CUTOFFS.max().to_pydatetime())


In [ ]:
daily_source["day"] = pd.to_datetime(daily_source["day"], utc=True)
calendar = pd.date_range(CUTOFFS.min() - LOOKBACK, CUTOFFS.max(), inclusive="left", freq="D")
daily_source = daily_source.set_index("day").reindex(calendar)
missing_days = daily_source[daily_source["rows"].isna()]
print("Дни без записанных сессий по всей базе:", len(missing_days))
display(missing_days)
display(daily_source.describe())
if not missing_days.empty:
    print("ВНИМАНИЕ: до проверки покрытия нельзя трактовать все пропуски как неактивность.")

if not (missing_days.empty):
    raise ValueError('Найдены дни без покрытия сессий: сначала уточните окна анализа')


В выбранных окнах сессии есть каждый день. Эта проверка не гарантирует полноту всех записей, но целых пропущенных дней не обнаружено.


## 4. Отбор подписок

Сначала загружу клиентов и их подписки, проверю даты начала и окончания. Сессии и участников добавлю отдельно, чтобы одна подписка случайно не превратилась в несколько строк.


In [ ]:
clients = q(f"""
    select client_id::text as client_id, registered_at,
           region_code, acquisition_channel
    from prime.clients where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id::text as subscription_id,
           owner_client_id::text as owner_client_id,
           tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT_SQL})
    order by subscription_id
""")


In [ ]:
for col in ["started_at", "ended_at"]:
    subs[col] = pd.to_datetime(subs[col], utc=True)
clients["registered_at"] = pd.to_datetime(clients["registered_at"], utc=True)

bad_sub_dates = subs[subs["ended_at"].notna() & (subs["ended_at"] < subs["started_at"])]
display(pd.DataFrame({"table": ["clients", "subs"], "rows": [len(clients), len(subs)]}))
print("Подписки с противоречивыми датами:", len(bad_sub_dates))
if not (bad_sub_dates.empty):
    raise ValueError('Сначала разберите противоречивые даты подписок')


Добавлю данные владельца, в том числе канал привлечения. Пропущенный канал оставлю отдельной категорией, чтобы такие подписки не исчезли из сравнения.


In [ ]:
base = subs.merge(clients, left_on="owner_client_id", right_on="client_id",
                  how="left", validate="m:1", indicator=True)
base = base.drop(columns="_merge")
base["acquisition_channel"] = base["acquisition_channel"].fillna("без канала")


На каждую дату оставлю подписки, которые уже начались, ещё действовали и существовали не менее 30 дней. Затем отмечу, какие из них закончились в следующие 30 дней.

Признак `is_trial` не использую для исключения: он говорит об оформлении с пробным периодом, а не о том, что пробный период продолжается на дату наблюдения.


In [ ]:
cohorts = []
inclusion_rows = []
for cutoff in CUTOFFS:
    active_mask = ((base["started_at"] < cutoff)
                   & (base["ended_at"].isna() | (base["ended_at"] >= cutoff)))
    mature_mask = base["started_at"] <= cutoff - LOOKBACK
    one = base.loc[active_mask & mature_mask].copy()
    one["cutoff"] = cutoff
    one["tenure_days"] = (cutoff - one["started_at"]).dt.total_seconds() / 86400
    one["churn_30d"] = (one["ended_at"].notna()
                        & (one["ended_at"] >= cutoff)
                        & (one["ended_at"] < cutoff + FOLLOWUP)).astype(int)
    cohorts.append(one)
    inclusion_rows.append({"cutoff": cutoff, "active": int(active_mask.sum()),
                           "excluded_age_under_30d": int((active_mask & ~mature_mask).sum()),
                           "included": len(one)})

panel_base = pd.concat(cohorts, ignore_index=True)
display(pd.DataFrame(inclusion_rows))
print("Объявлено строк будущей витрины:", len(panel_base))


## 5. Активность владельца

Для владельца каждой подписки посчитаю сессии, дни использования и число разных сервисов за предыдущие 30 дней. Повторные посещения одного сервиса не увеличивают число разных сервисов.

Учитываю сессии, которые начались в этом окне и корректно завершились к дате наблюдения. Ошибочные и ещё не завершённые сессии считаю отдельно.

В источнике сессии привязаны к человеку. Здесь измеряю активность владельца, поэтому его отсутствие сессий не означает, что подпиской не пользовались другие участники. Ниже отдельно рассмотрю владельцев без шеринга и участия в чужих подписках.


In [ ]:
activity_parts = []
for cutoff in CUTOFFS:
    activity = q(f"""
        with seg as ({SEGMENT_SQL}),
        sessions as (
            select u.*,
                   (u.ended_at > u.started_at and u.ended_at <= :cutoff) as valid,
                   exists (select 1 from prime.services v
                           where v.service_id = u.service_id) as known_service
            from prime.service_usage u
            join seg c on c.client_id = u.client_id
            where u.started_at >= :a and u.started_at < :cutoff
        )
        select client_id::text as owner_client_id,
               count(*) as raw_sessions,
               count(*) filter (where valid) as n_sessions,
               count(distinct started_at::date) filter (where valid) as active_days,
               count(distinct service_id) filter (where valid) as n_services,
               count(*) filter (where ended_at <= started_at) as invalid_sessions,
               count(*) filter (where ended_at is null or ended_at > :cutoff) as not_finished_at_cutoff,
               count(*) filter (where not known_service) as unknown_service_sessions
        from sessions group by client_id
    """, a=(cutoff - LOOKBACK).to_pydatetime(), cutoff=cutoff.to_pydatetime())
    activity["cutoff"] = cutoff
    activity_parts.append(activity)

activity = pd.concat(activity_parts, ignore_index=True)
if not (activity['unknown_service_sessions'].sum() == 0):
    raise ValueError('Найдены сессии неизвестных сервисов')


In [ ]:
# Отсутствие записей об использовании учитываю как нулевую активность.
panel = panel_base.merge(activity, on=["owner_client_id", "cutoff"],
                         how="left", validate="1:1", indicator=True)
print("Не нашли ни одной строки сессий в прошлом окне:", int((panel["_merge"] == "left_only").sum()))
panel = panel.drop(columns="_merge")
count_columns = ["raw_sessions", "n_sessions", "active_days", "n_services",
                 "invalid_sessions", "not_finished_at_cutoff", "unknown_service_sessions"]
panel[count_columns] = panel[count_columns].fillna(0).astype(int)


Разделю владельцев на три группы: без использования, с одним сервисом и с двумя или более. Отдельно отмечу случаи, когда записи есть, но все сессии непригодны для расчёта: таких людей нельзя уверенно считать неактивными.


In [ ]:
panel['activity_group'] = '2+ сервиса'
panel.loc[panel['n_services'] == 0, 'activity_group'] = '0 сервисов'
panel.loc[panel['n_services'] == 1, 'activity_group'] = '1 сервис'

# Записи есть, но по ним нельзя надёжно определить активность.
panel['only_unusable_sessions'] = (panel['raw_sessions'] > 0) & (panel['n_sessions'] == 0)
panel['tenure_group'] = pd.cut(
    panel['tenure_days'], bins=[29.999, 90, 180, np.inf],
    labels=['30–90 дней', '91–180 дней', '181+ дней']
)


In [ ]:
diagnostics = (panel.groupby(["cutoff", "activity_group"], observed=True, dropna=False)
               .agg(subscriptions=("subscription_id", "size"),
                    cancellations=("churn_30d", "sum"),
                    median_sessions=("n_sessions", "median"),
                    median_active_days=("active_days", "median"),
                    only_unusable_sessions=("only_unusable_sessions", "sum"))
               .reset_index())
display(diagnostics)


In [ ]:
# Объединение данных не должно менять число подписок и отмен.
audit = pd.DataFrame({
    "check": ["Объявлено строк", "Строк после соединения", "Уникальных подписка × дата",
              "Клиентов с несколькими действующими подписками на дату", "Дней без сессий в источнике"],
    "value": [len(panel_base), len(panel),
              len(panel.drop_duplicates(["subscription_id", "cutoff"])),
              int(panel.duplicated(["owner_client_id", "cutoff"]).sum()), len(missing_days)]
})
display(audit)

control = panel_base.groupby("cutoff")["churn_30d"].sum()
after_join = panel.groupby("cutoff")["churn_30d"].sum()
if not (control.equals(after_join)):
    raise ValueError('Соединение изменило число отмен')

print("Зерно и контроль отмен сохранены.")
print("Перейдите к расчётам ниже.")


## 6. Участники подписок и итоговая выборка

Владелец может оплачивать подписку для других людей и сам ей не пользоваться. Проверю, есть ли дополнительные участники и состоит ли владелец в другой подписке. Учитываю участие на дату наблюдения.

Эта проверка описывает состав участников. Суммарную активность всей семьи в основном анализе не рассчитываю.


In [ ]:
sharing_parts = []
for cutoff in CUTOFFS:
    sharing = q(f"""
        with seg as ({SEGMENT_SQL})
        select s.subscription_id::text as subscription_id,
               count(distinct m.client_id) as n_current_members,
               exists (
                   select 1 from prime.subscription_members other_m
                   where other_m.client_id = s.owner_client_id
                     and other_m.subscription_id <> s.subscription_id
                     and other_m.joined_at < :cutoff
                     and (other_m.left_at is null or other_m.left_at >= :cutoff)
               ) as owner_has_other_membership
        from prime.subscriptions s
        join seg c on c.client_id = s.owner_client_id
        left join prime.subscription_members m
          on m.subscription_id = s.subscription_id
         and m.joined_at < :cutoff
         and (m.left_at is null or m.left_at >= :cutoff)
        group by s.subscription_id, s.owner_client_id
    """, cutoff=cutoff.to_pydatetime())
    sharing['cutoff'] = cutoff
    sharing_parts.append(sharing)

sharing = pd.concat(sharing_parts, ignore_index=True)


In [ ]:
panel = panel.merge(
    sharing, on=['subscription_id', 'cutoff'], how='left',
    validate='1:1', indicator='_sharing_merge'
)
if (panel['_sharing_merge'] != 'both').any():
    raise ValueError('Для части подписок не загрузились данные об участниках')

panel = panel.drop(columns='_sharing_merge')
panel['n_current_members'] = panel['n_current_members'].astype(int)
panel['has_members'] = panel['n_current_members'] > 0

sharing_audit = panel.groupby('cutoff').agg(
    subscriptions=('subscription_id', 'size'),
    with_members=('has_members', 'sum'),
    owners_in_other_subscriptions=('owner_has_other_membership', 'sum'),
    only_unusable_sessions=('only_unusable_sessions', 'sum')
)
display(sharing_audit)


Исключу случаи, в которых все записанные сессии непригодны для расчёта. Затем проверю, меняется ли вывод при их возвращении.

На 1 июля действовали 19 084 подписки. После исключения 165 подписок со стажем меньше 30 дней осталось 18 919, затем исключены 12 случаев с непригодными сессиями. Итог — 18 907 подписок. Для июня — 19 360.

Также подготовлю группы по стажу подписки и числу сессий для сравнения похожих владельцев.


In [ ]:
analysis_panel = panel.loc[~panel['only_unusable_sessions']].copy()
analysis_panel['tenure_group'] = pd.cut(
    analysis_panel['tenure_days'], bins=[30, 90, 180, np.inf], right=False,
    labels=['30–89 дней', '90–179 дней', '180+ дней']
)
analysis_panel['session_band'] = pd.cut(
    analysis_panel['n_sessions'], bins=[1.5, 2.5, 3.5, 5.5, np.inf],
    labels=['2', '3', '4–5', '6+']
)
print('После исключения непригодных сессий, по датам:')
display(analysis_panel.groupby('cutoff').size().rename('subscriptions'))


## 7. Сравнение оттока

Отток — число подписок, закончившихся в следующие 30 дней, делённое на число подписок в группе на дату наблюдения. Сравниваю доли, поскольку размеры групп различаются.

В H1 сравниваю неактивных со всеми активными владельцами, в H2 — пользователей одного сервиса с пользователями нескольких. Разницу считаю как отток первой группы минус отток второй, в процентных пунктах. Положительное значение соответствует гипотезе.

Рядом с оценками показываю доверительные интервалы: Уилсона для долей и нормальное приближение для разницы. Если интервал разницы включает ноль, уверенно утверждать, что отток групп различается, нельзя. Это не доказывает отсутствие связи.


In [ ]:
Z95 = 1.959963984540054
Z975 = 2.241402727604947
GROUPS = ['0 сервисов', '1 сервис', '2+ сервиса']


In [ ]:
def wilson(events, total, z=Z95):
    
    if total == 0:
        return np.nan, np.nan

    share = events / total
    correction = 1 + z**2 / total
    center = (share + z**2 / (2 * total)) / correction
    margin = z * math.sqrt(share * (1 - share) / total + z**2 / (4 * total**2)) / correction
    return center - margin, center + margin


In [ ]:
def risk_table(data):
    table = data.groupby(['cutoff', 'activity_group'], observed=True).agg(
        n=('subscription_id', 'size'),
        events=('churn_30d', 'sum')
    ).reset_index()

    table['risk_pct'] = 100 * table['events'] / table['n']
    table['low_pct'] = 0.0
    table['high_pct'] = 0.0

    for i in table.index:
        lower, upper = wilson(table.loc[i, 'events'], table.loc[i, 'n'])
        table.loc[i, 'low_pct'] = lower * 100
        table.loc[i, 'high_pct'] = upper * 100

    return table


In [ ]:
def comparison_groups(data, hypothesis):
    if hypothesis == 'H1':
        first = data[data['n_services'] == 0]
        second = data[data['n_services'] >= 1]
    else:
        first = data[data['n_services'] == 1]
        second = data[data['n_services'] >= 2]
    return first, second


In [ ]:
def raw_comparison(data, hypothesis):
    first, second = comparison_groups(data, hypothesis)
    n1 = len(first)
    n2 = len(second)
    if n1 == 0 or n2 == 0:
        raise ValueError('Для сравнения нужны подписки в обеих группах')

    events1 = int(first['churn_30d'].sum())
    events2 = int(second['churn_30d'].sum())
    rate1 = events1 / n1
    rate2 = events2 / n2
    difference = rate1 - rate2
    error = math.sqrt(rate1 * (1 - rate1) / n1 + rate2 * (1 - rate2) / n2)

    status = 'ok'
    if min(events1, n1 - events1, events2, n2 - events2) < 10:
        status = 'малые числа: интервал приближённый'

    return {
        'hypothesis': hypothesis,
        'n_a': n1, 'events_a': events1,
        'n_b': n2, 'events_b': events2,
        'status': status,
        'risk_a_pct': rate1 * 100, 'risk_b_pct': rate2 * 100,
        'diff_pp': difference * 100,
        'low_pp': (difference - Z95 * error) * 100,
        'high_pp': (difference + Z95 * error) * 100
    }


In [ ]:
rates = risk_table(analysis_panel)
rates.round({'risk_pct': 3, 'low_pct': 3, 'high_pct': 3})


In [ ]:
raw_results = []
for cutoff, data in analysis_panel.groupby('cutoff'):
    for hypothesis in ['H1', 'H2']:
        row = {'cutoff': cutoff}
        row.update(raw_comparison(data, hypothesis))
        raw_results.append(row)

raw_results = pd.DataFrame(raw_results)
raw_results.round({'risk_a_pct': 3, 'risk_b_pct': 3, 'diff_pp': 3, 'low_pp': 3, 'high_pp': 3})


В июле отток пользователей нескольких сервисов ниже, но интервалы разницы включают ноль. В июне порядок групп другой. Проверю, не связаны ли различия с тарифом, каналом привлечения и стажем подписки.


## 8. Сравнение похожих владельцев

Группы могут различаться по тарифу, каналу привлечения и стажу. Например, меньший отток может быть связан с большим стажем, а не с использованием сервисов. Поэтому сравню владельцев внутри одинаковых сочетаний этих признаков. Стаж разделю на 30–89, 90–179 и 180+ дней.

Оставлю сочетания, где есть хотя бы по 20 подписок с каждой стороны. Покажу, какую долю исходных групп удалось сохранить.

Для H2 дополнительно учту число сессий: пользователи нескольких сервисов могут просто чаще пользоваться подпиской. Сравню владельцев с 2, 3, 4–5 и 6+ сессиями. Людей с одной сессией исключу из этой проверки: среди них не может быть пользователей двух сервисов. В широких диапазонах частота использования всё ещё может различаться.


In [ ]:
BASE_STRATA = ['tariff_id', 'acquisition_channel', 'tenure_group']
MIN_PER_GROUP = 20


In [ ]:
def prepare_strata(first, second, columns):
    
    left = first.groupby(columns, observed=True).agg(
        n=('churn_30d', 'size'), k=('churn_30d', 'sum')
    ).reset_index()
    right = second.groupby(columns, observed=True).agg(
        n=('churn_30d', 'size'), k=('churn_30d', 'sum')
    ).reset_index()

    return left.merge(right, on=columns, suffixes=('_a', '_b'))


Объединю сравнения с одинаковыми весами для обеих групп. Вес каждого сочетания — его доля в общей сопоставимой выборке. Так разный состав групп по выбранным признакам меньше влияет на итоговую разницу.

Интервалы приближённые, веса считаются фиксированными. Такой расчёт учитывает доступные признаки, но не устанавливает причину ухода.


In [ ]:
def weighted_rates(table):
    
    result = {}
    for column in ['risk_a_pct', 'risk_b_pct', 'diff_pp', 'low_pp', 'high_pp',
                   'low_975_pp', 'high_975_pp']:
        result[column] = np.nan
    if len(table) == 0:
        return result

    total = table['n_a'] + table['n_b']
    table['weight'] = total / total.sum()
    table['risk_a'] = table['k_a'] / table['n_a']
    table['risk_b'] = table['k_b'] / table['n_b']

    rate1 = (table['weight'] * table['risk_a']).sum()
    rate2 = (table['weight'] * table['risk_b']).sum()
    variance1 = table['risk_a'] * (1 - table['risk_a']) / table['n_a']
    variance2 = table['risk_b'] * (1 - table['risk_b']) / table['n_b']
    error = math.sqrt((table['weight']**2 * (variance1 + variance2)).sum())
    difference = rate1 - rate2

    result['risk_a_pct'] = rate1 * 100
    result['risk_b_pct'] = rate2 * 100
    result['diff_pp'] = difference * 100
    result['low_pp'] = (difference - Z95 * error) * 100
    result['high_pp'] = (difference + Z95 * error) * 100
    result['low_975_pp'] = (difference - Z975 * error) * 100
    result['high_975_pp'] = (difference + Z975 * error) * 100
    return result


Посчитаю итоговую разницу и долю подписок, оставшихся после отбора. Если осталось мало наблюдений, результат будет относиться лишь к небольшой части исходной группы.


In [ ]:
def standardized_comparison(data, hypothesis, strata, min_n=MIN_PER_GROUP):
    first, second = comparison_groups(data, hypothesis)
    original1 = len(first)
    original2 = len(second)
    if original1 == 0 or original2 == 0:
        raise ValueError('Для сравнения нужны подписки в обеих группах')

    if 'session_band' in strata:
        first = first[first['n_sessions'] >= 2]
        second = second[second['n_sessions'] >= 2]

    table = prepare_strata(first, second, strata)
    common = table[(table['n_a'] >= min_n) & (table['n_b'] >= min_n)].copy()
    n1 = int(common['n_a'].sum())
    n2 = int(common['n_b'].sum())

    result = {
        'hypothesis': hypothesis, 'strata': ' × '.join(strata),
        'n_original_a': original1, 'n_original_b': original2,
        'n_a': n1, 'n_b': n2, 'common_strata': len(common),
        'coverage_a_pct': n1 / original1 * 100,
        'coverage_b_pct': n2 / original2 * 100,
        'coverage_pct': (n1 + n2) / (original1 + original2) * 100,
        'status': 'ok'
    }
    result.update(weighted_rates(common))

    if len(common) == 0:
        result['status'] = 'нет общего покрытия'
    elif min(result['coverage_a_pct'], result['coverage_b_pct']) < 70:
        result['status'] = 'ограниченное покрытие: вывод только для сопоставимой части'

    if len(common) > 0:
        event_counts = [common['k_a'].sum(), common['k_b'].sum(),
                        (common['n_a'] - common['k_a']).sum(), (common['n_b'] - common['k_b']).sum()]
        if min(event_counts) < 10:
            result['status'] += '; мало исходов: интервал ненадёжен'

    return result, common


In [ ]:
settings = [
    ('H1', 'тариф + канал + стаж', BASE_STRATA),
    ('H2', 'тариф + канал + стаж', BASE_STRATA),
    ('H2', 'тариф + канал + стаж + число сессий', BASE_STRATA + ['session_band'])
]

adjusted_rows = []
strata_details = {}
for cutoff, data in analysis_panel.groupby('cutoff'):
    for hypothesis, specification, strata in settings:
        estimate, detail = standardized_comparison(data, hypothesis, strata)
        row = {'cutoff': cutoff, 'specification': specification}
        row.update(estimate)
        adjusted_rows.append(row)
        strata_details[(cutoff, hypothesis, specification)] = detail

adjusted = pd.DataFrame(adjusted_rows)
adjusted[['cutoff', 'hypothesis', 'specification', 'n_a', 'n_b', 'common_strata',
                  'coverage_a_pct', 'coverage_b_pct', 'diff_pp', 'low_pp', 'high_pp', 'status']].round({
    'coverage_a_pct':3,	'coverage_b_pct': 3,	'diff_pp': 3,	'low_pp': 3, 	'high_pp': 3})


Основных гипотез две, поэтому дополнительно покажу интервалы 97,5% с поправкой Бонферрони. Они шире обычных 95%-х и учитывают две основные проверки.


In [ ]:
is_july = adjusted['cutoff'] == PRIMARY_CUTOFF
is_h1 = adjusted['hypothesis'] == 'H1'
with_sessions = adjusted['specification'].str.contains('число сессий')
primary_adjusted = adjusted[is_july & (is_h1 | with_sessions)].copy()

print('Две основные скорректированные оценки: интервалы 97,5% для двух гипотез')
display(primary_adjusted[['hypothesis', 'specification', 'coverage_a_pct', 'coverage_b_pct',
                         'diff_pp', 'low_975_pp', 'high_975_pp', 'status']].round(3))


Для H1 в июле осталось около 98% обеих групп. Для H2 после учёта числа сессий — лишь 4,9% пользователей одного сервиса и 20,3% пользователей нескольких. Поэтому результат H2 нельзя переносить на всех активных владельцев. Широкий интервал также не позволяет сделать уверенный вывод.


## 9. Дополнительные проверки

Проверю, насколько результат зависит от отбора: верну случаи с непригодными сессиями, оставлю подписки со стажем от 90 дней и отдельно рассмотрю владельцев без участников и без участия в чужих подписках.

Это дополнительные сравнения. Находка в одной подгруппе сама по себе не подтверждает основную гипотезу.


In [ ]:
primary = analysis_panel[analysis_panel['cutoff'] == PRIMARY_CUTOFF].copy()
raw_primary = panel[panel['cutoff'] == PRIMARY_CUTOFF].copy()
without_sharing = primary[
    (primary['n_current_members'] == 0) & (primary['owner_has_other_membership'] == False)
]
variants = {
    'основная выборка': primary,
    'включены непригодные сессии': raw_primary,
    'стаж от 90 дней': primary[primary['tenure_days'] >= 90],
    'без шеринга и чужих подписок': without_sharing
}

sensitivity_rows = []
for name, data in variants.items():
    for hypothesis in ['H1', 'H2']:
        row = {'sample': name}
        row.update(raw_comparison(data, hypothesis))
        sensitivity_rows.append(row)

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.round(3)


Для H2 попробую пороги 10, 20 и 30 подписок с каждой стороны. Это покажет, можно ли получить более широкую сопоставимую выборку при другом пороге. Отметка покрытия ниже 70% служит предупреждением о потере наблюдений, а не статистическим критерием.


In [ ]:
support_sensitivity = []
for min_n in [10, 20, 30]:
    estimate, detail = standardized_comparison(primary, 'H2', BASE_STRATA + ['session_band'], min_n)
    row = {'min_per_group': min_n}
    row.update(estimate)
    support_sensitivity.append(row)

support_sensitivity = pd.DataFrame(support_sensitivity)
support_sensitivity[['min_per_group', 'common_strata', 'coverage_a_pct',
                             'coverage_b_pct', 'diff_pp', 'low_pp', 'high_pp', 'status']].round(3)


У владельцев без участников и участия в чужих подписках отток составил 5,12% для одного сервиса и 2,53% для нескольких. Разница заметная, но здесь всего 50 отмен, а тариф, стаж, канал и частота использования в этом сравнении не учтены.

Этот результат стоит проверить на новом периоде. Обычный 95%-й интервал не учитывает, что дополнительных сравнений было несколько.


## 10. За счёт чего изменился общий отток

Общий отток может измениться по двум причинам: изменились доли трёх групп или отток внутри самих групп.

Разделю эти вклады методом цепных подстановок. Сначала рассчитаю, каким был бы отток при июльских долях групп и июньском оттоке внутри них. Разница с фактическим июнем — вклад состава. Затем заменю отток внутри групп на июльский и получу второй вклад.

Общий отток: **C = Σ w_g × q_g**, где w — доля группы, q — её отток. Вклады:

- состав: **Σ (w₁ − w₀) × q₀**;
- отток внутри групп: **Σ w₁ × (q₁ − q₀)**.

Индекс 0 — июнь, 1 — июль. Для результата в процентных пунктах вклады умножаются на 100. Их сумма должна совпасть с общим изменением.

Группы формируются заново каждый месяц. Разложение показывает, где изменился показатель, но не доказывает, что дополнительные сервисы удержали конкретных людей.


In [ ]:
first_cutoff = CUTOFFS.min()
last_cutoff = PRIMARY_CUTOFF

def decomposition(data, first, last):
    old_data = data[data['cutoff'] == first]
    new_data = data[data['cutoff'] == last]
    old = old_data.groupby('activity_group', observed=True)['churn_30d'].agg(['size', 'sum'])
    new = new_data.groupby('activity_group', observed=True)['churn_30d'].agg(['size', 'sum'])
    old = old.reindex(GROUPS)
    new = new.reindex(GROUPS)
    if old['size'].isna().any() or new['size'].isna().any():
        raise ValueError('В одном из периодов нет группы для разложения')

    detail = pd.DataFrame()
    detail['n0'] = old['size']
    detail['n1'] = new['size']
    detail['q0'] = old['sum'] / old['size']
    detail['q1'] = new['sum'] / new['size']
    detail['w0'] = detail['n0'] / detail['n0'].sum()
    detail['w1'] = detail['n1'] / detail['n1'].sum()

    detail['composition_pp'] = (detail['w1'] - detail['w0']) * detail['q0'] * 100
    detail['within_group_pp'] = detail['w1'] * (detail['q1'] - detail['q0']) * 100
    total_old = old_data['churn_30d'].mean()
    total_new = new_data['churn_30d'].mean()
    actual_pp = (total_new - total_old) * 100

    effects_sum = detail['composition_pp'].sum() + detail['within_group_pp'].sum()
    if abs(effects_sum - actual_pp) > 0.00000001:
        raise ValueError('Сумма вкладов не совпала с изменением оттока')
    return detail, total_old, total_new, actual_pp


In [ ]:
factor_table, total_old, total_new, actual_pp = decomposition(analysis_panel, first_cutoff, last_cutoff)
display(factor_table.round(5))
factor_totals = factor_table[['composition_pp', 'within_group_pp']].sum()
print(f'Общий отток: {total_old:.3%} → {total_new:.3%}; изменение {actual_pp:+.4f} п.п.')
print(f'Состав групп: {factor_totals["composition_pp"]:+.4f} п.п.')
print(f'Отток внутри групп: {factor_totals["within_group_pp"]:+.4f} п.п.')
print('Контроль суммы вкладов пройден.')


Повторю расчёт в обратном порядке: сначала изменю отток внутри групп, затем их доли. Суммарное изменение должно совпасть; распределение вкладов может немного отличаться.


In [ ]:
reverse_composition = 100*((factor_table['w1']-factor_table['w0'])*factor_table['q1']).sum()
reverse_within = 100*(factor_table['w0']*(factor_table['q1']-factor_table['q0'])).sum()
if not (abs(reverse_composition + reverse_within - actual_pp) < 1e-10):
    raise ValueError('Проверка не пройдена')
pd.DataFrame({'порядок': ['состав → отток', 'отток → состав'],
                      'состав_пп': [factor_totals['composition_pp'], reverse_composition],
                      'отток_пп': [factor_totals['within_group_pp'], reverse_within]}).round(5)


## 11. Графики

Сравню июльский отток трёх групп. На графике указаны размеры групп и 95%-е интервалы для долей.


In [ ]:
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 12, 'font.size': 10})

plot_rates = rates[(rates['cutoff'] == PRIMARY_CUTOFF)].set_index('activity_group').reindex(GROUPS)
fig1, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
positions = np.arange(len(GROUPS))
ax.bar(positions, plot_rates['risk_pct'], color=['#9CA3AF', '#4B5563', '#277DA8'])
ax.errorbar(positions, plot_rates['risk_pct'],
            yerr=[plot_rates['risk_pct']-plot_rates['low_pct'],
                  plot_rates['high_pct']-plot_rates['risk_pct']],
            fmt='none', color='#111827', capsize=5)
for x, (_, row) in zip(positions, plot_rates.iterrows()):
    ax.text(x, row['high_pct']+0.1, f'{row["risk_pct"]:.2f}%\nn={int(row["n"]):,}', ha='center')
ax.set_xticks(positions, GROUPS)
ax.set_ylim(0, plot_rates['high_pct'].max()+0.8)
ax.set_ylabel('Отток за следующие 30 дней, %')
ax.set_title(f'В июле отток групп — от {plot_rates["risk_pct"].min():.2f}% '
             f'до {plot_rates["risk_pct"].max():.2f}%', loc='left')
ax.set_xlabel('Активность владельца за предыдущие 30 дней; интервалы Уилсона 95%; UTC')
ax.grid(axis='y', alpha=.15)
plt.show()


Покажу разницу оттока после учёта остальных признаков. Пунктирная линия обозначает нулевую разницу. Подписи показывают, какая доля каждой группы осталась в сравнении.


In [ ]:
forest = adjusted[(adjusted['cutoff'] == PRIMARY_CUTOFF)].copy()
forest = forest[forest['diff_pp'].notna()].reset_index(drop=True)
fig2, ax = plt.subplots(figsize=(10, 4.3), layout='constrained')
if forest.empty:
    ax.text(.5, .5, 'Нет достаточного общего покрытия для стандартизации',
            ha='center', va='center', transform=ax.transAxes)
else:
    labels = []
    for i, row in forest.iterrows():
        ax.errorbar(row['diff_pp'], i,
                    xerr=[[row['diff_pp']-row['low_pp']],
                          [row['high_pp']-row['diff_pp']]],
                    fmt='o', capsize=4, color='#277DA8')
        control_label = '+ число сессий' if 'число сессий' in row['specification'] else 'тариф, канал, стаж'
        labels.append(f'{row["hypothesis"]}: {control_label}\n'
                      f'Покрытие групп: {row["coverage_a_pct"]:.0f}% / {row["coverage_b_pct"]:.0f}%')
    ax.set_yticks(np.arange(len(forest)), labels)
    ax.invert_yaxis()
ax.axvline(0, color='#6B7280', linestyle='--')
uncertain = not forest.empty and ((forest['low_pp'] <= 0) & (forest['high_pp'] >= 0)).all()
ax.set_title('После поправки различия остаются неопределёнными' if uncertain
             else 'Сопоставление групп после учёта наблюдаемых различий', loc='left')
ax.set_xlabel('Разница оттока, п.п.; 95%-е приближённые интервалы; основная дата — 01.07.2026')
ax.grid(axis='x', alpha=.15)
plt.show()


Сравню результаты июня и июля. Некоторые подписки встречаются в обоих периодах, поэтому считаю периоды отдельно и не объединяю их в одну выборку независимых наблюдений.


In [ ]:
time_rates = rates.pivot(index='cutoff', columns='activity_group', values='risk_pct').reindex(columns=GROUPS)
fig3, ax = plt.subplots(figsize=(8, 4.3), layout='constrained')
month_labels = []
for date in time_rates.index:
    month_labels.append(date.strftime('%d.%m.%Y'))
for group, color in zip(GROUPS, ['#9CA3AF', '#4B5563', '#277DA8']):
    ax.plot(month_labels, time_rates[group], marker='o', label=group, color=color)
first_order = time_rates.iloc[0].sort_values().index.tolist()
last_order = time_rates.iloc[-1].sort_values().index.tolist()
same_order = first_order == last_order
ax.set_title('Порядок групп по оттоку сохраняется в двух периодах' if same_order
             else 'Порядок групп по оттоку меняется между июнем и июлем', loc='left')
ax.set_ylim(bottom=0)
ax.set_ylabel('Отток за следующие 30 дней, %')
ax.set_xlabel('Дата наблюдения; активность — за предыдущие 30 дней; UTC')
ax.legend(title='Активность владельца', frameon=False)
ax.grid(axis='y', alpha=.15)
plt.show()


На следующем графике — вклад изменения состава и оттока внутри групп в общее изменение между июнем и июлем.


In [ ]:
deltas = factor_totals.rename(index={'composition_pp': 'Изменение\nсостава групп',
                                      'within_group_pp': 'Изменение оттока\nвнутри групп'})
starts = deltas.cumsum()-deltas
fig4, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
bar_colors = []
for value in deltas:
    if value > 0:
        bar_colors.append('#E09F3E')
    else:
        bar_colors.append('#277DA8')
ax.bar(range(2), deltas, bottom=starts, color=bar_colors)
ax.bar(2, actual_pp, color='#6B7280')
for i, value in enumerate(deltas):
    endpoint = starts.iloc[i] + value
    ax.annotate(f'{value:+.4f}', (i, endpoint), xytext=(0, 8 if value >= 0 else -14),
                textcoords='offset points', ha='center')
ax.annotate(f'{actual_pp:+.4f}', (2, actual_pp), xytext=(0, 8 if actual_pp >= 0 else -14),
            textcoords='offset points', ha='center')
ax.set_xticks([0, 1, 2], [*deltas.index, 'Общее\nизменение'])
ax.axhline(0, color='#111827', linewidth=.8)
ax.set_ylabel('Вклад в изменение оттока, п.п.')
ax.set_title(f'Общий отток изменился на {actual_pp:+.4f} п.п. между июнем и июлем', loc='left')
ax.set_xlabel('Сначала состав групп, затем отток внутри; окна отмен — по 30 суток; UTC')
levels = [0] + starts.tolist() + deltas.cumsum().tolist() + [actual_pp]
span = max(max(levels) - min(levels), .001)
ax.set_ylim(min(levels)-.35*span, max(levels)+.35*span)
ax.grid(axis='y', alpha=.15)
plt.show()


## 12. Выводы и рекомендации

Я сравнил отток пользователей с разной активностью. В июле он составил 3,80% в группе без активности, 3,94% среди пользователей одного сервиса и 3,45% среди пользователей нескольких. На первый взгляд, использование нескольких сервисов связано с меньшим оттоком. Но разница оказалась недостаточно убедительной, а в июне такая картина не повторилась. Поэтому уверенно подтвердить гипотезы пока нельзя.

Я также сравнил людей с одинаковым тарифом, каналом привлечения и похожим стажем подписки. Это не изменило вывод по первой гипотезе. Для второй я дополнительно учёл количество сессий, но после отбора осталось слишком мало сопоставимых пользователей. Поэтому вопрос о связи разнообразия сервисов с оттоком остаётся открытым.

Общий отток немного снизился: с 3,827% в июне до 3,713% в июле. При разложении этого изменения я выяснил, что главный вклад внесло снижение оттока среди пользователей нескольких сервисов. Изменение долей групп почти не повлияло на результат. Чтобы понять, сохраняется ли снижение, нужно посмотреть следующие месяцы.

Я бы пока не рекомендовал выбирать аудиторию кампании только по отсутствию активности или использованию одного сервиса. Сначала стоит повторить анализ на новом периоде. Особенно интересно проверить подгруппу без шеринга: там отток пользователей одного сервиса был 5,12%, а нескольких - 2,53%. Разница заметная, но требует дополнительной проверки.

Следующим шагом я предлагаю провести эксперимент: предложить части пользователей попробовать второй сервис, а случайно выбранной контрольной группе ничего не менять. Через 30 дней сравнить отток и посчитать, окупились ли затраты на предложение. Если кампания помогает сохранять подписки и приносит больше дополнительной маржи, чем стоит, её можно расширять.
